#### PyTorch 와 RNN, LSTM

input, sequence, 하이퍼 파라미터 설정

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import numpy as np
from copy import deepcopy

In [1]:
sequence_length = 28 # MNIST row 를 일종의 순서(sequence) 로 다름
feature_size = 28 # 입력 차원
hidden_size = 128 # Hidden Layer 사이즈 설정처럼 설정
num_layers = 4 # stacked RNN (최대 4개까지는 Gradient Vanishing 현상이 적을 수 있을므로)
dropout_p = 0.2
output_size = 10 # 0 ~ 9 숫자 분류 (클래스)
minibatch_size = 128

In [7]:
class Net(nn.Module):
    def __init__(self, feature_size, hidden_size, num_layers, dropout_p, output_size, model_type):
        super().__init__()
        if model_type == 'rnn':
            self.sequenceclassifier = nn.RNN(
                input_size = feature_size,
                hidden_size = hidden_size,
                num_layers = num_layers,
                batch_first = True,
                dropout = dropout_p,
                bidirectional = True
            )
        elif model_type == 'lstm':
            self.sequenceclassifier = nn.LSTM(
                input_size = feature_size,
                hidden_size = hidden_size,
                num_layers = num_layers,
                batch_first = True,
                dropout = dropout_p,
                bidirectional = True
            )
        
        self.liner_layers = nn.Sequential(
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(hidden_size * 2),
            nn.Linear(2 * 128, output_size),
            nn.LogSoftmax(dim=-1)
        )
        
    def forward(self, x):
        out, _  = self.sequenceclassifier(x)
        out = out[: , -1]
        print(out)
        return self.liner_layers(out)

In [ ]:
data1 = torch.full((minibatch_size, sequence_length, 2 * hidden_size), 1) # vector 생성 (128, 28, 2 * 128)
data2 = data1[:, -1] # (128, 256)
print(data1.shape, data2.shape)
data3 = torch.full((minibatch_size, 1, sequence_length, feature_size), 1) # vector 생성 (128, 1, 28 , 28)
data4 = data3.reshape(-1, sequence_length, feature_size) # (128, 28 ,28)
print(data3.shape, data4.shape)

torch.Size([128, 28, 256]) torch.Size([128, 256])
torch.Size([128, 1, 28, 28]) torch.Size([128, 28, 28])


In [8]:
model = Net(feature_size, hidden_size, num_layers, dropout_p, output_size, 'rnn')
model

Net(
  (sequenceclassifier): RNN(28, 128, num_layers=4, batch_first=True, dropout=0.2, bidirectional=True)
  (liner_layers): Sequential(
    (0): LeakyReLU(negative_slope=0.1)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Linear(in_features=256, out_features=10, bias=True)
    (3): LogSoftmax(dim=-1)
  )
)